In [0]:
!pip install unitycatalog-ai[databricks]==0.3.2
%restart_python

In [0]:
%run ../../Includes/_common

In [0]:
from unitycatalog.ai.core.databricks import DatabricksFunctionClient

In [0]:
# Create a python DA object from the dbacademy.ops.meta table
DA = DBAcademyHelper()
DA.init()

In [0]:
def use_uc_env():
    catalog_name = DA.catalog_name
    schema_name = DA.schema_name 

    spark.sql(f"USE CATALOG {DA.catalog_name}")
    spark.sql(f"USE SCHEMA {DA.schema_name}")
    return catalog_name, schema_name 

In [0]:
def process_airbnb_dataset(databricks_share_name: str):
    # Read the CSV file from the volume with headers
    df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .option("multiLine", "true") \
        .option("escape", '"') \
        .load(f"/Volumes/{databricks_share_name}/v01/sf-listings/sf-airbnb.csv")

    # Write as a Delta table
    df.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable("sf_airbnb_listings")
        
    print(f"✅ Successfully created table {DA.catalog_name}.{DA.schema_name}.sf_airbnb_listings.")

In [0]:
# Set UC environment
catalog_name, schema_name = use_uc_env()

# Process the Airbnb dataset
process_airbnb_dataset(
        databricks_share_name = "dbacademy_airbnb"
        )

In [0]:
def set_environment_and_tools(catalog_name:str, schema_name:str) -> None:
    """
    Sets the environment and tools for the notebook.
    """

    query1 = f"""
    USE CATALOG {catalog_name}
    """
    query2 = f"""
    USE SCHEMA {schema_name}
    """
    query3 = """
    DROP FUNCTION IF EXISTS avg_neigh_price
    """
    query4 = """
    DROP FUNCTION IF EXISTS airbnb_posting_info
    """
    query5 = f"""
    CREATE OR REPLACE FUNCTION avg_neigh_price(
    neighborhood_name STRING COMMENT "The neighborhood name to filter by (e.g., 'Mission', 'Upper Market')"
    )
    RETURNS DOUBLE
    LANGUAGE SQL
    DETERMINISTIC
    COMMENT 'Calculates the average listing price for a specific neighborhood in San Francisco. Returns the average price as a numeric value. Price strings are cleaned and converted to numeric values before averaging.'
    RETURN 
    SELECT AVG(CAST(REGEXP_REPLACE(price, '[^0-9.]', '') AS DOUBLE))
    FROM sf_airbnb_listings
    WHERE neighbourhood_cleansed = neighborhood_name
    AND price IS NOT NULL
    AND REGEXP_REPLACE(price, '[^0-9.]', '') != ''
    """
    print(f"Using catalog `{catalog_name}` and schema `{schema_name}`")
    spark.sql(query1)
    spark.sql(query2)
    
    print("Creating functions...")
    spark.sql(query3).collect()
    spark.sql(query4).collect()
    
    spark.sql(query5).collect()
    print(f"Created function avg_price_by_neighborhood")

    def airbnb_posting_info(id: int) -> str:
        """
        Fetches Airbnb posting information as formatted text.

        Args:
            id (int): Airbnb listing ID (e.g., 958)

        Returns:
            str: Formatted listing information (description, reviews, and rating) or error message
        """
        import requests
        import re

        api_url = f"https://www.airbnb.com/rooms/{id}"
        
        try:
            response = requests.get(api_url, timeout=10)
            
            if response.status_code == 200:
                html = response.text
                
                # Extract description
                desc = re.search(r'"metaDescription":"([^"]+)"', html)
                if desc:
                    description = desc.group(1).replace('\\n', ' ')
                    parts = description.split(' · ')
                    description = ' · '.join(parts[2:]) if len(parts) > 2 else description
                else:
                    description = "Description not found"
                
                # Extract review count and rating
                reviews = re.search(r'"reviewCount":(\d+)', html)
                rating = re.search(r'"starRating":([\d.]+)', html)
                
                reviews = reviews.group(1) if reviews else "N/A"
                rating = rating.group(1) if rating else "N/A"
                
                return f"""Description: {description}

    Reviews: {reviews}
    Rating: {rating} stars"""
            else:
                return f'Request failed with status code: {response.status_code}'
        
        except requests.exceptions.RequestException as e:
            return f'Request error: {str(e)}'


    # client = DatabricksFunctionClient() # For classic compute
    client = DatabricksFunctionClient(execution_mode="serverless") # For serverless compute
    function_info = client.create_python_function(
    func=airbnb_posting_info,
    catalog=catalog_name,
    schema=schema_name,
    replace=True
    )
    return None

In [0]:
set_environment_and_tools(catalog_name, schema_name)